# Hyperspectral Super-Resolution: EMIT → Regression-Synthetic S2

**Goal:** Train a neural network to spatially super-resolve EMIT hyperspectral imagery (60 m) to 10 m resolution, using regression-synthetic Sentinel-2 tiles as ground truth.

| | LR (input) | HR / GT (target) |
|---|---|---|
| **Source** | EMIT (32-band subset) | Regression-synthetic S2 (via `S2ToEMITRegressor`) |
| **Resolution** | 60 m | 10 m |
| **Bands** | 32 | 32 |
| **Tile size** | 120 × 120 px | 720 × 720 px |
| **Scale factor** | — | 6× |

## Pipeline
1. Install BasicSR + dependencies
2. Score tiles by R² and build a quality manifest (filter out clouds/corruption)
3. Prepare `.npy` tile pairs from manifest-selected GeoTIFF tiles
4. Register a custom multi-band dataset loader
5. Register a modified RRDBNet that handles 6× upsampling and 32 channels
6. Configure, train, evaluate

In [ ]:
# ============================================================
# Cell 1 — Setup
# ============================================================
# Clone BasicSR, install dependencies

!git clone https://github.com/XPixelGroup/BasicSR.git 2>/dev/null || echo 'BasicSR already cloned'
%cd /content/BasicSR
!pip install -e . -q
!pip install rasterio joblib scikit-learn -q

import sys
sys.path.insert(0, '/content/BasicSR')

# Add project root so we can import spectral/, tiles_helpers/, etc.
sys.path.insert(0, '/content/drive/MyDrive/HyperSpectralSuperResolution')

In [ ]:
# ============================================================
# Cell 2 — Data preparation: manifest-driven LR + GT tiles → .npy
# ============================================================
# Reads the manifest.csv produced by Cell 1b to select only
# high-quality tiles (R² ≥ threshold). Works for regression-synth
# GT; for SFIM, just swap in a different manifest.
#
# LR = EMIT-b32 tiles (32 bands, 120×120 at 60m)  — uint16
# GT = regression-synth tiles (32 bands, 720×720 at 10m) — uint16
#
# Both are normalized to float32 reflectance (÷10000) before saving.

import csv
import re
from pathlib import Path
import numpy as np
import rasterio
from tqdm import tqdm

# ================= CONFIG =================
MANIFEST_PATH = Path("/content/drive/MyDrive/run/manifest.csv")  # from Cell 1b

OUT_DIR = Path("/content/data_basicsr_emit_s2")

# Split config
VAL_FRACTION  = 0.15
TEST_FRACTION = 0.10

# Normalization
DN_SCALE      = 10000.0
NODATA_U16    = 65535
# ==========================================


def _read_tile(path, is_uint16_scaled=True):
    """Read a GeoTIFF tile → (C, H, W) float32 reflectance."""
    with rasterio.open(path) as ds:
        arr = ds.read().astype(np.float32)
    if is_uint16_scaled:
        mask = (arr >= NODATA_U16 - 0.5)
        arr /= DN_SCALE
        arr[mask] = 0.0
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = np.clip(arr, 0.0, None)
    return arr


def _save_pair(lr, gt, stem, split):
    """Validate shapes and save as .npy."""
    assert lr.shape[0] == gt.shape[0], \
        f"Band mismatch: LR {lr.shape} vs GT {gt.shape}"
    expected_scale = gt.shape[1] / lr.shape[1]
    assert abs(expected_scale - 6.0) < 0.01, \
        f"Scale mismatch: LR {lr.shape} → GT {gt.shape} = {expected_scale:.1f}×"
    np.save(OUT_DIR / split / 'lq' / f"{stem}.npy", lr)
    np.save(OUT_DIR / split / 'gt' / f"{stem}.npy", gt)


def prepare_data():
    # Read manifest
    with open(MANIFEST_PATH) as f:
        reader = csv.DictReader(f)
        rows = [r for r in reader if r["keep"] == "1"]

    if not rows:
        raise FileNotFoundError(
            f"No kept tiles in {MANIFEST_PATH}. "
            "Lower MIN_R2 in Cell 1b or check your data."
        )

    print(f"Manifest: {len(rows)} tiles pass R² filter")

    # Create output dirs
    for split in ['train', 'val', 'test']:
        (OUT_DIR / split / 'lq').mkdir(parents=True, exist_ok=True)
        (OUT_DIR / split / 'gt').mkdir(parents=True, exist_ok=True)

    # Quick peek at first pair
    r0 = rows[0]
    with rasterio.open(r0["emit_b32_path"]) as ds:
        lr_shape = (ds.count, ds.height, ds.width)
    with rasterio.open(r0["synth_path"]) as ds:
        gt_shape = (ds.count, ds.height, ds.width)
    print(f"  Example: LR {lr_shape}, GT {gt_shape}, "
          f"scale={gt_shape[1]/lr_shape[1]:.0f}×, "
          f"R²={r0['r2_mean']}")

    # Random split
    n = len(rows)
    rng = np.random.default_rng(42)
    idxs = rng.permutation(n)
    n_test = max(1, int(n * TEST_FRACTION))
    n_val  = max(1, int(n * VAL_FRACTION))
    test_set = set(idxs[:n_test])
    val_set  = set(idxs[n_test:n_test + n_val])

    print(f"  Split: train {n - n_val - n_test}, val {n_val}, test {n_test}\n")

    for i, row in enumerate(tqdm(rows, desc="Preparing .npy")):
        if i in test_set:
            split = 'test'
        elif i in val_set:
            split = 'val'
        else:
            split = 'train'

        lr = _read_tile(row["emit_b32_path"], is_uint16_scaled=True)
        gt = _read_tile(row["synth_path"],    is_uint16_scaled=True)

        stem = f"{row['aoi']}__{row['pair']}__tile{int(row['tile_id']):03d}"
        _save_pair(lr, gt, stem, split)

    print(f"\nDone → {OUT_DIR}")
    for split in ['train', 'val', 'test']:
        cnt = len(list((OUT_DIR / split / 'lq').glob('*.npy')))
        print(f"  {split}: {cnt} pairs")


prepare_data()

In [ ]:
# ============================================================
# Cell 2 — Data preparation: pre-computed LR + GT tiles → .npy
# ============================================================
# Your pipeline already produced:
#   - EMIT-b32 tiles (LR):  32 bands, 100×100, 60 m
#   - Regression-synth tiles (GT): 32 bands, 600×600, 10 m (via S2ToEMITRegressor)
#
# Each tile was fitted with its own per-tile .joblib regressor,
# so we don't need the regressor here — just read the finished GeoTIFFs.
#
# FILE NAMING:
#   GT:  {S2_timestamp}_{S2_tile}_{EMIT_date}_tile{NNN}_regression_synth.tif
#   LR:  tile_{NNN}_emit_b32.tif   (← ADJUST this pattern if different!)
#
# The pairing key is the tile number (NNN) extracted from both filenames.
#
# SPLITTING — two modes:
#   Mode A: single folder, random split by tile (quick experiments)
#   Mode B: per-scene folders, hold out entire scenes (proper evaluation)

import re
from pathlib import Path
import numpy as np
import rasterio
from tqdm import tqdm

# ================= CONFIG — EDIT THESE =================

# Folders containing your pre-computed tiles
GT_TILES_DIR = Path("/content/drive/MyDrive/regression_synth_tiles")   # *_regression_synth.tif
LR_TILES_DIR = Path("/content/drive/MyDrive/emit_b32_tiles") # *_emit_b32.tif

# If LR and GT are in the SAME folder, set both to the same path:
# GT_TILES_DIR = LR_TILES_DIR = Path("/content/drive/MyDrive/all_tiles")

# Glob patterns — adjust if your naming is different
GT_GLOB = "*_regression_synth.tif"    # regression-synthetic EMIT at 10 m
LR_GLOB = "*_emit_b32.tif"           # EMIT 32-band subset at 60 m

OUT_DIR = Path("/content/data_basicsr_emit_s2")

# How to extract the tile number from filenames (used for pairing)
def _tile_id_from_gt(name):
    """Extract tile number from GT filename like '..._tile282_regression_synth.tif'."""
    m = re.search(r'tile(\d+)', name)
    return int(m.group(1)) if m else None

def _tile_id_from_lr(name):
    """Extract tile number from LR filename like 'tile_282_emit_b32.tif'."""
    m = re.search(r'tile[_]?(\d+)', name)
    return int(m.group(1)) if m else None

# Split config
VAL_FRACTION  = 0.15
TEST_FRACTION = 0.10
EMIT_SCALE    = 10000.0    # uint16 → reflectance (if LR is uint16)
EMIT_NODATA_U16 = 65535

# --- Mode B (optional): per-scene split ---
# If your tiles come from multiple scenes in separate folders,
# use this dict instead.  Set to None for random split (Mode A).
#
# SCENE_DIRS = {
#     'grandcanyon': {
#         'gt': Path("/content/drive/MyDrive/grandcanyon/regression_synth"),
#         'lr': Path("/content/drive/MyDrive/grandcanyon/b32"),
#     },
#     'la': {
#         'gt': Path("/content/drive/MyDrive/la/regression_synth"),
#         'lr': Path("/content/drive/MyDrive/la/b32"),
#     },
# }
# VAL_SCENES  = ['la']
# TEST_SCENES = []
SCENE_DIRS  = None
VAL_SCENES  = []
TEST_SCENES = []
# =======================================================


def _read_tile(path, is_uint16_scaled=False):
    """Read a GeoTIFF tile → (C, H, W) float32."""
    with rasterio.open(path) as ds:
        arr = ds.read().astype(np.float32)
    if is_uint16_scaled:
        mask = (arr >= EMIT_NODATA_U16 - 0.5)
        arr /= EMIT_SCALE
        arr[mask] = 0.0
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = np.clip(arr, 0.0, None)
    return arr


def _save_pair(lr, gt, stem, split):
    """Validate shapes and save as .npy."""
    assert lr.shape[0] == gt.shape[0], \
        f"Band mismatch: LR {lr.shape} vs GT {gt.shape}"
    expected_scale = gt.shape[1] / lr.shape[1]
    assert abs(expected_scale - 6.0) < 0.01, \
        f"Scale mismatch: LR {lr.shape} → GT {gt.shape} = {expected_scale:.1f}×"
    np.save(OUT_DIR / split / 'lq' / f"{stem}.npy", lr)
    np.save(OUT_DIR / split / 'gt' / f"{stem}.npy", gt)


def _pair_tiles(gt_dir, lr_dir):
    """Find GT/LR pairs by matching tile IDs. Returns list of (gt_path, lr_path, tile_id)."""
    gt_files = {_tile_id_from_gt(p.name): p for p in Path(gt_dir).glob(GT_GLOB)}
    lr_files = {_tile_id_from_lr(p.name): p for p in Path(lr_dir).glob(LR_GLOB)}

    # Remove None keys (files that didn't match the regex)
    gt_files.pop(None, None)
    lr_files.pop(None, None)

    common_ids = sorted(set(gt_files) & set(lr_files))
    gt_only = set(gt_files) - set(lr_files)
    lr_only = set(lr_files) - set(gt_files)

    if gt_only:
        print(f"  {len(gt_only)} GT tiles with no matching LR: {sorted(gt_only)[:5]}...")
    if lr_only:
        print(f"  {len(lr_only)} LR tiles with no matching GT: {sorted(lr_only)[:5]}...")

    return [(gt_files[tid], lr_files[tid], tid) for tid in common_ids]


def prepare_data():
    for split in ['train', 'val', 'test']:
        (OUT_DIR / split / 'lq').mkdir(parents=True, exist_ok=True)
        (OUT_DIR / split / 'gt').mkdir(parents=True, exist_ok=True)

    # ------ Mode B: scene-level split ------
    if SCENE_DIRS is not None:
        counts = {'train': 0, 'val': 0, 'test': 0}
        for scene_name, dirs in SCENE_DIRS.items():
            if scene_name in TEST_SCENES:
                split = 'test'
            elif scene_name in VAL_SCENES:
                split = 'val'
            else:
                split = 'train'

            pairs = _pair_tiles(dirs['gt'], dirs['lr'])
            print(f"Scene '{scene_name}' -> {split}  ({len(pairs)} pairs)")

            for gt_path, lr_path, tid in tqdm(pairs, desc=f"  {scene_name}", leave=False):
                lr = _read_tile(lr_path, is_uint16_scaled=True)
                gt = _read_tile(gt_path, is_uint16_scaled=False)
                _save_pair(lr, gt, f"{scene_name}__tile{tid:03d}", split)
                counts[split] += 1

        print(f"\nDone (scene-level split):")
        for s, c in counts.items():
            print(f"  {s}: {c} pairs")
        return

    # ------ Mode A: single source, random split ------
    pairs = _pair_tiles(GT_TILES_DIR, LR_TILES_DIR)
    if not pairs:
        print(f"GT dir:  {GT_TILES_DIR}  ->  {list(GT_TILES_DIR.glob(GT_GLOB))[:3]}")
        print(f"LR dir:  {LR_TILES_DIR}  ->  {list(LR_TILES_DIR.glob(LR_GLOB))[:3]}")
        raise FileNotFoundError(
            "No matched tile pairs found. Check GT_TILES_DIR, LR_TILES_DIR, "
            "and the glob/regex patterns above."
        )

    # Quick peek at the first pair
    gt0, lr0, tid0 = pairs[0]
    with rasterio.open(gt0) as ds:
        gt_shape = (ds.count, ds.height, ds.width)
    with rasterio.open(lr0) as ds:
        lr_shape = (ds.count, ds.height, ds.width)
    print(f"Found {len(pairs)} paired tiles")
    print(f"  Example pair (tile {tid0}):")
    print(f"    LR: {lr0.name}  shape {lr_shape}")
    print(f"    GT: {gt0.name}  shape {gt_shape}")
    print(f"    Scale: {gt_shape[1]/lr_shape[1]:.1f}x\n")

    n = len(pairs)
    rng = np.random.default_rng(42)
    idxs = rng.permutation(n)
    n_test = max(1, int(n * TEST_FRACTION))
    n_val  = max(1, int(n * VAL_FRACTION))
    test_set = set(idxs[:n_test])
    val_set  = set(idxs[n_test:n_test + n_val])

    print(f"Split: train {n - n_val - n_test}, val {n_val}, test {n_test}")

    for i, (gt_path, lr_path, tid) in enumerate(tqdm(pairs, desc="Preparing")):
        if i in test_set:
            split = 'test'
        elif i in val_set:
            split = 'val'
        else:
            split = 'train'

        lr = _read_tile(lr_path, is_uint16_scaled=True)
        gt = _read_tile(gt_path, is_uint16_scaled=False)
        _save_pair(lr, gt, f"tile{tid:03d}", split)

    print(f"\nDone:")
    for split in ['train', 'val', 'test']:
        cnt = len(list((OUT_DIR / split / 'lq').glob('*.npy')))
        print(f"  {split}: {cnt} pairs")


prepare_data()

In [ ]:
# ============================================================
# Cell 4 — Register RRDBNet6x (modified for 6× upsampling)
# ============================================================
# BasicSR's default RRDBNet does 2× + 2× = 4× upsampling.
# We need 2× + 3× = 6× for EMIT (60 m) → S2 (10 m).

import torch
import torch.nn as nn
import torch.nn.functional as F
from basicsr.archs.rrdbnet_arch import RRDB  # reuse the RRDB block
from basicsr.utils.registry import ARCH_REGISTRY


@ARCH_REGISTRY.register()
class RRDBNet6x(nn.Module):
    """RRDBNet adapted for 6× upsampling (2× then 3×).

    Architecture is identical to the standard RRDBNet except:
    - First upsample: 2× nearest-neighbour interpolation + conv
    - Second upsample: 3× nearest-neighbour interpolation + conv
    - Total: 6× spatial upsampling

    Supports arbitrary num_in_ch / num_out_ch (e.g. 32 for hyperspectral).
    """

    def __init__(
        self,
        num_in_ch,
        num_out_ch,
        num_feat=64,
        num_block=23,
        num_grow_ch=32,
    ):
        super().__init__()
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)

        # RRDB trunk
        self.body = nn.ModuleList()
        for _ in range(num_block):
            self.body.append(RRDB(num_feat, num_grow_ch=num_grow_ch))
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)

        # Upsample: 2× then 3× = 6×
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)  # after 2×
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)  # after 3×

        # Output
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(x)

        # RRDB trunk with residual
        body_feat = feat
        for block in self.body:
            body_feat = block(body_feat)
        body_feat = self.conv_body(body_feat)
        feat = feat + body_feat

        # Upsample 2×
        feat = F.interpolate(feat, scale_factor=2, mode='nearest')
        feat = self.lrelu(self.conv_up1(feat))

        # Upsample 3×
        feat = F.interpolate(feat, scale_factor=3, mode='nearest')
        feat = self.lrelu(self.conv_up2(feat))

        # Output projection
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return out


# Quick sanity check
_net = RRDBNet6x(num_in_ch=32, num_out_ch=32, num_feat=64, num_block=4)
_x  = torch.randn(1, 32, 20, 20)  # 120/6 = 20 LR pixels
_y  = _net(_x)
print(f"Input: {_x.shape} → Output: {_y.shape}")
assert _y.shape == (1, 32, 120, 120), f"Expected (1,32,120,120), got {_y.shape}"
n_params = sum(p.numel() for p in _net.parameters()) / 1e6
print(f"Params (4 blocks): {n_params:.1f}M")

# Full model size estimate
_net23 = RRDBNet6x(num_in_ch=32, num_out_ch=32, num_feat=64, num_block=23)
n_params_full = sum(p.numel() for p in _net23.parameters()) / 1e6
print(f"Params (23 blocks): {n_params_full:.1f}M")
del _net, _net23, _x, _y

print("RRDBNet6x registered.")

In [ ]:
# ============================================================
# Cell 5 — Generate training config YAML
# ============================================================

import yaml
from pathlib import Path

# -------- CONFIG — EDIT THESE --------
DATA_ROOT  = '/content/data_basicsr_emit_s2'
EXP_NAME   = 'EMIT_S2_SR_x6_RRDBNet'
NUM_BANDS  = 32
SCALE      = 6
GT_SIZE    = 120    # crop size in GT domain (120 / 6 = 20 LR pixels)
BATCH_SIZE = 4
TOTAL_ITER = 100_000
LR_RATE    = 2e-4
NUM_BLOCK  = 16     # fewer blocks than default 23 — saves memory, train faster
NUM_FEAT   = 64
# -------------------------------------

config = {
    'name': EXP_NAME,
    'model_type': 'SRModel',
    'scale': SCALE,
    'num_gpu': 1,
    'manual_seed': 42,

    'datasets': {
        'train': {
            'name': 'EMIT_S2_train',
            'type': 'PairedNpyDataset',
            'dataroot_gt': f'{DATA_ROOT}/train/gt',
            'dataroot_lq': f'{DATA_ROOT}/train/lq',
            'gt_size': GT_SIZE,
            'scale': SCALE,
            'use_hflip': True,
            'use_rot': True,
            'num_worker_per_gpu': 4,
            'batch_size_per_gpu': BATCH_SIZE,
            'dataset_enlarge_ratio': 100,
        },
        'val': {
            'name': 'EMIT_S2_val',
            'type': 'PairedNpyDataset',
            'dataroot_gt': f'{DATA_ROOT}/val/gt',
            'dataroot_lq': f'{DATA_ROOT}/val/lq',
            'gt_size': GT_SIZE,
            'scale': SCALE,
            'io_backend': {'type': 'disk'},
        },
    },

    'network_g': {
        'type': 'RRDBNet6x',
        'num_in_ch': NUM_BANDS,
        'num_out_ch': NUM_BANDS,
        'num_feat': NUM_FEAT,
        'num_block': NUM_BLOCK,
        'num_grow_ch': 32,
    },

    'path': {
        'pretrain_network_g': None,
        'strict_load_g': True,
        'resume_state': None,
    },

    'train': {
        'ema_decay': 0.999,
        'optim_g': {
            'type': 'Adam',
            'lr': LR_RATE,
            'weight_decay': 0,
            'betas': [0.9, 0.99],
        },
        'scheduler': {
            'type': 'MultiStepLR',
            'milestones': [25000, 50000, 75000],
            'gamma': 0.5,
        },
        'total_iter': TOTAL_ITER,
        'warmup_iter': -1,
        'pixel_opt': {
            'type': 'L1Loss',
            'loss_weight': 1.0,
            'reduction': 'mean',
        },
    },

    'val': {
        'val_freq': 2000,
        'save_img': False,  # 32-band images can't be saved as PNG
        'metrics': {
            'psnr': {
                'type': 'calculate_psnr',
                'crop_border': SCALE,
                'test_y_channel': False,
            },
        },
    },

    'logger': {
        'print_freq': 100,
        'save_checkpoint_freq': 5000,
        'use_tb_logger': True,
    },

    'dist_params': {
        'backend': 'nccl',
        'port': 29500,
    },
}

# Write YAML
config_path = Path('/content/BasicSR/options/train') / f'{EXP_NAME}.yml'
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config written to: {config_path}")
print(f"\nKey settings:")
print(f"  Bands: {NUM_BANDS}")
print(f"  Scale: {SCALE}×")
print(f"  GT crop: {GT_SIZE}×{GT_SIZE} → LR crop: {GT_SIZE//SCALE}×{GT_SIZE//SCALE}")
print(f"  Network: RRDBNet6x, {NUM_BLOCK} blocks, {NUM_FEAT} features")
print(f"  Total iterations: {TOTAL_ITER:,}")
print(f"  Batch size: {BATCH_SIZE}")

In [ ]:
# ============================================================
# Cell 5 — Generate training config YAML
# ============================================================

import yaml
from pathlib import Path

# -------- CONFIG — EDIT THESE --------
DATA_ROOT  = '/content/data_basicsr_emit_s2'
EXP_NAME   = 'EMIT_S2_SR_x6_RRDBNet'
NUM_BANDS  = 32
SCALE      = 6
GT_SIZE    = 192    # crop size in GT domain (192 / 6 = 32 LR pixels)
BATCH_SIZE = 4
TOTAL_ITER = 100_000
LR_RATE    = 2e-4
NUM_BLOCK  = 16     # fewer blocks than default 23 — saves memory, train faster
NUM_FEAT   = 64
# -------------------------------------

config = {
    'name': EXP_NAME,
    'model_type': 'SRModel',
    'scale': SCALE,
    'num_gpu': 1,
    'manual_seed': 42,

    'datasets': {
        'train': {
            'name': 'EMIT_S2_train',
            'type': 'PairedNpyDataset',
            'dataroot_gt': f'{DATA_ROOT}/train/gt',
            'dataroot_lq': f'{DATA_ROOT}/train/lq',
            'gt_size': GT_SIZE,
            'scale': SCALE,
            'use_hflip': True,
            'use_rot': True,
            'num_worker_per_gpu': 4,
            'batch_size_per_gpu': BATCH_SIZE,
            'dataset_enlarge_ratio': 100,
        },
        'val': {
            'name': 'EMIT_S2_val',
            'type': 'PairedNpyDataset',
            'dataroot_gt': f'{DATA_ROOT}/val/gt',
            'dataroot_lq': f'{DATA_ROOT}/val/lq',
            'gt_size': GT_SIZE,
            'scale': SCALE,
            'io_backend': {'type': 'disk'},
        },
    },

    'network_g': {
        'type': 'RRDBNet6x',
        'num_in_ch': NUM_BANDS,
        'num_out_ch': NUM_BANDS,
        'num_feat': NUM_FEAT,
        'num_block': NUM_BLOCK,
        'num_grow_ch': 32,
    },

    'path': {
        'pretrain_network_g': None,
        'strict_load_g': True,
        'resume_state': None,
    },

    'train': {
        'ema_decay': 0.999,
        'optim_g': {
            'type': 'Adam',
            'lr': LR_RATE,
            'weight_decay': 0,
            'betas': [0.9, 0.99],
        },
        'scheduler': {
            'type': 'MultiStepLR',
            'milestones': [25000, 50000, 75000],
            'gamma': 0.5,
        },
        'total_iter': TOTAL_ITER,
        'warmup_iter': -1,
        'pixel_opt': {
            'type': 'L1Loss',
            'loss_weight': 1.0,
            'reduction': 'mean',
        },
    },

    'val': {
        'val_freq': 2000,
        'save_img': False,  # 32-band images can't be saved as PNG
        'metrics': {
            'psnr': {
                'type': 'calculate_psnr',
                'crop_border': SCALE,
                'test_y_channel': False,
            },
        },
    },

    'logger': {
        'print_freq': 100,
        'save_checkpoint_freq': 5000,
        'use_tb_logger': True,
    },

    'dist_params': {
        'backend': 'nccl',
        'port': 29500,
    },
}

# Write YAML
config_path = Path('/content/BasicSR/options/train') / f'{EXP_NAME}.yml'
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config written to: {config_path}")
print(f"\nKey settings:")
print(f"  Bands: {NUM_BANDS}")
print(f"  Scale: {SCALE}×")
print(f"  GT crop: {GT_SIZE}×{GT_SIZE} → LR crop: {GT_SIZE//SCALE}×{GT_SIZE//SCALE}")
print(f"  Network: RRDBNet6x, {NUM_BLOCK} blocks, {NUM_FEAT} features")
print(f"  Total iterations: {TOTAL_ITER:,}")
print(f"  Batch size: {BATCH_SIZE}")

In [ ]:
# ============================================================
# Cell 6 — Train
# ============================================================
# NOTE: The custom dataset and architecture must be registered
# BEFORE calling train.py. Since we registered them in cells 3-4,
# we call the training function directly (not via subprocess).

import sys
sys.path.insert(0, '/content/BasicSR')

from basicsr.train import train_pipeline
import basicsr.train as train_module

# The registrations from cells 3 and 4 are already active in this
# Python session. We just need to call the training pipeline.

config_path = '/content/BasicSR/options/train/EMIT_S2_SR_x6_RRDBNet.yml'

# Override sys.argv so BasicSR's argparser sees our config
sys.argv = ['train.py', '-opt', config_path]

# Launch training (this blocks until done or interrupted)
import importlib
importlib.reload(train_module)

# If train_pipeline exists and is callable:
try:
    train_pipeline(train_module.parse_options(is_train=True))
except Exception as e:
    # Fallback: run as subprocess with the registrations injected
    print(f"Direct call failed ({e}), falling back to subprocess...")
    print("NOTE: For subprocess, you need to put the dataset + arch")
    print("      registrations into .py files in BasicSR/basicsr/")
    raise

# ============================================================
# Cell 7 — Inference + Visualisation
# ============================================================
# Load the trained model and run inference on a validation tile.

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# -------- CONFIG --------
CHECKPOINT = '/content/BasicSR/experiments/EMIT_S2_SR_x6_RRDBNet/models/net_g_latest.pth'
VAL_LQ_DIR = Path('/content/data_basicsr_emit_s2/val/lq')
VAL_GT_DIR = Path('/content/data_basicsr_emit_s2/val/gt')
NUM_BANDS  = 32
SCALE      = 6
# Select 3 bands for RGB visualisation (indices into the 32-band subset)
# These approximate true-color from the EMIT wavelength range
VIS_BANDS  = [10, 6, 2]  # R, G, B — adjust based on your b32 wavelengths
# -------------------------


def load_model(checkpoint_path):
    """Load trained RRDBNet6x from checkpoint."""
    net = RRDBNet6x(
        num_in_ch=NUM_BANDS, num_out_ch=NUM_BANDS,
        num_feat=64, num_block=16, num_grow_ch=32,
    )
    state = torch.load(checkpoint_path, map_location='cpu')
    # BasicSR saves under 'params_ema' or 'params'
    if 'params_ema' in state:
        net.load_state_dict(state['params_ema'], strict=True)
    elif 'params' in state:
        net.load_state_dict(state['params'], strict=True)
    else:
        net.load_state_dict(state, strict=True)
    net.eval()
    return net


def percentile_stretch(img, lo=2, hi=98):
    """Per-channel percentile stretch to [0, 1]."""
    out = np.empty_like(img, dtype=np.float32)
    for c in range(img.shape[2]):
        v = img[:, :, c]
        p_lo = np.percentile(v[v > 0], lo) if np.any(v > 0) else 0
        p_hi = np.percentile(v[v > 0], hi) if np.any(v > 0) else 1
        if p_hi <= p_lo:
            p_hi = p_lo + 1e-6
        out[:, :, c] = np.clip((v - p_lo) / (p_hi - p_lo), 0, 1)
    return out


def make_rgb(cube_chw, bands=VIS_BANDS):
    """Extract 3 bands from (C,H,W) → (H,W,3) and stretch."""
    rgb = np.stack([cube_chw[b] for b in bands], axis=-1)
    return percentile_stretch(rgb)


# Load model
net = load_model(CHECKPOINT)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = net.to(device)

# Pick a validation tile
val_files = sorted(VAL_LQ_DIR.glob('*.npy'))
if not val_files:
    print("No validation tiles found.")
else:
    lq_path = val_files[0]
    gt_path = VAL_GT_DIR / lq_path.name

    lq = np.load(lq_path).astype(np.float32)  # (32, 120, 120)
    gt = np.load(gt_path).astype(np.float32)  # (32, 720, 720)

    # Inference
    with torch.no_grad():
        lq_tensor = torch.from_numpy(lq[None]).float().to(device)
        sr_tensor = net(lq_tensor)
        sr = sr_tensor.squeeze(0).cpu().numpy()

    # Bilinear baseline (for comparison — matches WALD's protocol)
    import torch.nn.functional as F_torch
    bilinear = F_torch.interpolate(
        torch.from_numpy(lq[None]).float(),
        scale_factor=SCALE, mode='bilinear', align_corners=False,
    ).squeeze(0).numpy()

    # Visualise
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(make_rgb(lq))
    axes[0].set_title(f'LR (EMIT 60m)\n{lq.shape[1]}x{lq.shape[2]}')

    axes[1].imshow(make_rgb(bilinear))
    axes[1].set_title(f'Bilinear x{SCALE}\n{bilinear.shape[1]}x{bilinear.shape[2]}')

    axes[2].imshow(make_rgb(sr))
    axes[2].set_title(f'SR (RRDBNet6x)\n{sr.shape[1]}x{sr.shape[2]}')

    axes[3].imshow(make_rgb(gt))
    axes[3].set_title(f'GT (regression synth 10m)\n{gt.shape[1]}x{gt.shape[2]}')

    for ax in axes:
        ax.axis('off')

    plt.suptitle(f'{lq_path.stem} — bands {VIS_BANDS}', fontsize=14)
    plt.tight_layout()
    plt.show()

    # Per-band metrics
    from skimage.metrics import peak_signal_noise_ratio as psnr

    psnrs_sr = []
    psnrs_bil = []
    for b in range(NUM_BANDS):
        gt_b = gt[b]
        sr_b = sr[b]
        bil_b = bilinear[b]
        data_range = gt_b.max() - gt_b.min()
        if data_range > 0:
            psnrs_sr.append(psnr(gt_b, sr_b, data_range=data_range))
            psnrs_bil.append(psnr(gt_b, bil_b, data_range=data_range))

    print(f"\nMean PSNR  —  SR: {np.mean(psnrs_sr):.2f} dB  |  "
          f"Bilinear: {np.mean(psnrs_bil):.2f} dB  |  "
          f"Gain: {np.mean(psnrs_sr) - np.mean(psnrs_bil):+.2f} dB")

In [ ]:
# ============================================================
# Cell 7 — Inference + Visualisation
# ============================================================
# Load the trained model and run inference on a validation tile.

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# -------- CONFIG --------
CHECKPOINT = '/content/BasicSR/experiments/EMIT_S2_SR_x6_RRDBNet/models/net_g_latest.pth'
VAL_LQ_DIR = Path('/content/data_basicsr_emit_s2/val/lq')
VAL_GT_DIR = Path('/content/data_basicsr_emit_s2/val/gt')
NUM_BANDS  = 32
SCALE      = 6
# Select 3 bands for RGB visualisation (indices into the 32-band subset)
# These approximate true-color from the EMIT wavelength range
VIS_BANDS  = [10, 6, 2]  # R, G, B — adjust based on your b32 wavelengths
# -------------------------


def load_model(checkpoint_path):
    """Load trained RRDBNet6x from checkpoint."""
    net = RRDBNet6x(
        num_in_ch=NUM_BANDS, num_out_ch=NUM_BANDS,
        num_feat=64, num_block=16, num_grow_ch=32,
    )
    state = torch.load(checkpoint_path, map_location='cpu')
    # BasicSR saves under 'params_ema' or 'params'
    if 'params_ema' in state:
        net.load_state_dict(state['params_ema'], strict=True)
    elif 'params' in state:
        net.load_state_dict(state['params'], strict=True)
    else:
        net.load_state_dict(state, strict=True)
    net.eval()
    return net


def percentile_stretch(img, lo=2, hi=98):
    """Per-channel percentile stretch to [0, 1]."""
    out = np.empty_like(img, dtype=np.float32)
    for c in range(img.shape[2]):
        v = img[:, :, c]
        p_lo = np.percentile(v[v > 0], lo) if np.any(v > 0) else 0
        p_hi = np.percentile(v[v > 0], hi) if np.any(v > 0) else 1
        if p_hi <= p_lo:
            p_hi = p_lo + 1e-6
        out[:, :, c] = np.clip((v - p_lo) / (p_hi - p_lo), 0, 1)
    return out


def make_rgb(cube_chw, bands=VIS_BANDS):
    """Extract 3 bands from (C,H,W) → (H,W,3) and stretch."""
    rgb = np.stack([cube_chw[b] for b in bands], axis=-1)
    return percentile_stretch(rgb)


# Load model
net = load_model(CHECKPOINT)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = net.to(device)

# Pick a validation tile
val_files = sorted(VAL_LQ_DIR.glob('*.npy'))
if not val_files:
    print("No validation tiles found.")
else:
    lq_path = val_files[0]
    gt_path = VAL_GT_DIR / lq_path.name

    lq = np.load(lq_path).astype(np.float32)  # (32, 100, 100)
    gt = np.load(gt_path).astype(np.float32)  # (32, 600, 600)

    # Inference
    with torch.no_grad():
        lq_tensor = torch.from_numpy(lq[None]).float().to(device)  # (1, 32, 100, 100)
        sr_tensor = net(lq_tensor)  # (1, 32, 600, 600)
        sr = sr_tensor.squeeze(0).cpu().numpy()  # (32, 600, 600)

    # Bicubic baseline (for comparison)
    import torch.nn.functional as F_torch
    bicubic = F_torch.interpolate(
        torch.from_numpy(lq[None]).float(),
        scale_factor=SCALE, mode='bicubic', align_corners=False,
    ).squeeze(0).numpy()

    # Visualise
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(make_rgb(lq))
    axes[0].set_title(f'LR (EMIT 60m)\n{lq.shape[1]}x{lq.shape[2]}')

    axes[1].imshow(make_rgb(bicubic))
    axes[1].set_title(f'Bicubic x{SCALE}\n{bicubic.shape[1]}x{bicubic.shape[2]}')

    axes[2].imshow(make_rgb(sr))
    axes[2].set_title(f'SR (RRDBNet6x)\n{sr.shape[1]}x{sr.shape[2]}')

    axes[3].imshow(make_rgb(gt))
    axes[3].set_title(f'GT (regression synth 10m)\n{gt.shape[1]}x{gt.shape[2]}')

    for ax in axes:
        ax.axis('off')

    plt.suptitle(f'{lq_path.stem} — bands {VIS_BANDS}', fontsize=14)
    plt.tight_layout()
    plt.show()

    # Per-band metrics
    from skimage.metrics import peak_signal_noise_ratio as psnr
    from skimage.metrics import structural_similarity as ssim

    # Compute PSNR per band
    psnrs_sr = []
    psnrs_bic = []
    for b in range(NUM_BANDS):
        gt_b = gt[b]
        sr_b = sr[b]
        bic_b = bicubic[b]
        data_range = gt_b.max() - gt_b.min()
        if data_range > 0:
            psnrs_sr.append(psnr(gt_b, sr_b, data_range=data_range))
            psnrs_bic.append(psnr(gt_b, bic_b, data_range=data_range))

    print(f"\nMean PSNR  —  SR: {np.mean(psnrs_sr):.2f} dB  |  "
          f"Bicubic: {np.mean(psnrs_bic):.2f} dB  |  "
          f"Gain: {np.mean(psnrs_sr) - np.mean(psnrs_bic):+.2f} dB")

In [ ]:
# ============================================================
# Cell 8 — Spectral fidelity analysis
# ============================================================
# Compare per-band accuracy: SR output vs GT across all 32 bands.

import numpy as np
import matplotlib.pyplot as plt

# Reuse sr, gt, bicubic from the previous cell

def spectral_metrics(pred, gt_cube):
    """Compute per-band RMSE and R² between predicted and GT cubes."""
    n_bands = pred.shape[0]
    rmse = np.zeros(n_bands)
    r2   = np.zeros(n_bands)
    for b in range(n_bands):
        p = pred[b].ravel()
        g = gt_cube[b].ravel()
        valid = np.isfinite(p) & np.isfinite(g) & (g > 0)
        if valid.sum() < 10:
            rmse[b] = np.nan
            r2[b] = np.nan
            continue
        p, g = p[valid], g[valid]
        rmse[b] = np.sqrt(np.mean((p - g) ** 2))
        ss_res = np.sum((p - g) ** 2)
        ss_tot = np.sum((g - g.mean()) ** 2)
        r2[b] = 1 - ss_res / max(ss_tot, 1e-10)
    return rmse, r2


rmse_sr, r2_sr = spectral_metrics(sr, gt)
rmse_bic, r2_bic = spectral_metrics(bicubic, gt)

bands = np.arange(NUM_BANDS)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(bands, rmse_bic, 'o-', label='Bicubic', alpha=0.7)
ax1.plot(bands, rmse_sr,  's-', label='SR (RRDBNet6x)', alpha=0.7)
ax1.set_xlabel('Band index')
ax1.set_ylabel('RMSE (reflectance)')
ax1.set_title('Per-band RMSE')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(bands, r2_bic, 'o-', label='Bicubic', alpha=0.7)
ax2.plot(bands, r2_sr,  's-', label='SR (RRDBNet6x)', alpha=0.7)
ax2.set_xlabel('Band index')
ax2.set_ylabel('R²')
ax2.set_title('Per-band R²')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.suptitle('Spectral Fidelity: SR vs Bicubic', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Mean R² — SR: {np.nanmean(r2_sr):.4f} | Bicubic: {np.nanmean(r2_bic):.4f}")
print(f"Mean RMSE — SR: {np.nanmean(rmse_sr):.6f} | Bicubic: {np.nanmean(rmse_bic):.6f}")